# Feature Engineering

In [ ]:
PROJECT_ID = "delta-vertex-513114-v2" #Puck
#PROJECT_ID = "winged-odyssey-513110-s3" #Fay

DATASET_PROJECT_ID = 'amsterdamumcdb'
DATASET_ID = 'version1_5_0'
LOCATION = 'eu'

In [ ]:
import os
import google.auth
import pandas as pd

#brew install --cask google-cloud-sdk   # skip if already installedd
#gcloud auth application-default login
#gcloud auth application-default set-quota-project liquid-folio-513114-d9


# all libraries check this environment variable, so set it:
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID

credentials, _ = google.auth.default()
print('Authenticated')

In [ ]:
%load_ext bigquery_magics
from bigquery_magics import bigquery_magics
from google.cloud import bigquery

# sets the default query job configuration
def_config = bigquery.job.QueryJobConfig(default_dataset=DATASET_PROJECT_ID + "." + DATASET_ID)
bigquery_magics.context.default_query_job_config = def_config

In [ ]:
from google.cloud import bigquery

# BigQuery requires a separate config to prevent the 'BadRequest: 400 Cannot explicitly modify anonymous table' error message
job_config = bigquery.job.QueryJobConfig()

# sets default client settings by re-using the previously defined config
client = bigquery.Client(project=PROJECT_ID, location=LOCATION, default_query_job_config=def_config)

In [ ]:
from cohort import getCohort

goldStandard = getCohort()

#check if columns are present and if there are duplicates
assert {"person_id", "visit_occurrence_id", "attemptNumber", "extubationTime"}.issubset(goldStandard.columns)

#remove dublicaes
assert not goldStandard.duplicated(
    ["visit_occurrence_id", "attemptNumber"]
).any()

#make the cohort for the DB BigQuery query
cohortRowList = []

for row in goldStandard.itertuples():
    cohortRowList.append(
        f"STRUCT({row.visit_occurrence_id} AS visitId, "
        f"{row.attemptNumber} AS attemptNumber, "
        f"TIMESTAMP '{row.extubationTime.isoformat()}' AS extubationTime)"
    )

cohortRows = ", ".join(cohortRowList)

# Time window before extubation
HOURS_BEFORE = 24

print("Extubation attempts:", len(goldStandard))

# Feature engineering - Pulmonary

In [ ]:
#sputum amount vs color, including source values.
sql = f"""
SELECT observation_source_value, value_source_value, value_as_concept_id,
       COUNT(*) AS n_records, COUNT(DISTINCT person_id) AS n_patients
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.observation`
WHERE provider_id IS NOT NULL
  AND observation_concept_id = 4089228
  AND observation_source_value IN ('Hoeveelheid Sputum', 'Kleur Sputum')
GROUP BY 1, 2, 3
ORDER BY 1, n_records DESC
"""
sputum_value_audit = client.query(sql).to_dataframe()
display(sputum_value_audit)

#### Interpretation of concept id 4089228: sputum
Devide into two features: colour and amout of sputum
Amount: Geen, Gering, Normaal, Ruim, Veel.
Color: Wit, Geel, Purulent, Groen, etc.

In [ ]:
#sputum consistency values.
sql = f"""
SELECT measurement_source_value, value_source_value, value_as_concept_id,
       COUNT(*) AS n_records, COUNT(DISTINCT person_id) AS n_patients
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE provider_id IS NOT NULL AND measurement_concept_id = 4080582
GROUP BY 1, 2, 3
ORDER BY n_records DESC
"""
consistency_value_audit = client.query(sql).to_dataframe()
display(consistency_value_audit)

#### Interpretation of concept id 4080582: sputum_consistency
Devide into 1 feature with 3 answers: Waterig, Kleverig, Taai.

In [ ]:
#airway suctioning quantity distribution and source labels.
sql = f"""
SELECT procedure_source_value, quantity,
       COUNT(*) AS n_records, COUNT(DISTINCT person_id) AS n_patients
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence`
WHERE provider_id IS NOT NULL
  AND procedure_concept_id IN (4239191, 4043669, 4336460)
GROUP BY 1, 2
ORDER BY 1, n_records DESC
"""
suctioning_value_audit = client.query(sql).to_dataframe()
display(suctioning_value_audit)

#### Interpretation of concept id 4239191, 4043669, 4336460: airway suctioning
The procedure Aantal Bronchiaaltoilet (concept ID 4239191) contains a numeric quantity value ranging from 1 to 8. --> after inspection in the DB conclusion is that one quantity represents a seperate registration period and is not comulative yet.

So feature: airway_suctioning_count_24h where we take the sum of recorded quantities within 24 hours before extubation.


In [ ]:
#Check pneumonia diagnoses

pneumonia_ids = [255848, 4309106, 257315, 4049965, 261326, 4190647]

sql = f"""
SELECT
    condition_concept_id,
    condition_source_value,
    COUNT(*) AS n_records,
    COUNT(DISTINCT person_id) AS n_patients

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence`

WHERE provider_id IS NOT NULL
    AND condition_concept_id IN ({", ".join(map(str, pneumonia_ids))})

GROUP BY 1, 2
ORDER BY n_records DESC
"""

display(client.query(sql).to_dataframe())


#### Interpretation of the concept ids for pneumonia
Different kind of pneumonia: bacterieel, viraal, aspiratie, schimmel en overige pneumonie. Some source registrations also refer to pulmonary sepsis.

In [ ]:
#Check COPD diagnosis registrations

sql = f"""
SELECT
    condition_source_value,
    COUNT(*) AS n_records,
    COUNT(DISTINCT person_id) AS n_patients

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence`

WHERE provider_id IS NOT NULL
    AND condition_concept_id = 257004

GROUP BY 1
ORDER BY n_records DESC
"""

display(client.query(sql).to_dataframe())


#### Interpretation of the concept id for COPD
The concept ID (257004) includes source registrations describing chronic obstructive pulmonary disease and emphysema/bronchitis, although the standardized concept refers to an acute COPD exacerbation.
This feature will be a COPD related diagnosis. 

Because COPD is a chronic condition, a diagnosis recorded before the current ICU admission may also be clinically relevant.

In [ ]:
#mechanical ventilation registrations

sql = f"""
SELECT
    measurement_source_value,
    value_source_value,
    value_as_concept_id,
    unit_source_value,
    COUNT(*) AS n_records,
    MIN(value_as_number) AS min_value,
    MAX(value_as_number) AS max_value

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

WHERE provider_id IS NOT NULL
    AND measurement_concept_id = 3004921

GROUP BY 1, 2, 3, 4
ORDER BY n_records DESC
"""

display(client.query(sql).to_dataframe())


#### interpretation of mechanical ventillation
Concept ID 3004921 contains categorical registrations of ventilator modes, including PS/CPAP, PC, VC and NAVA. The recorded values describe the selected ventilation mode. 
ventilation_mode is the latest recorded ventilator mode  24 hours before extubation. The original value_source_value is used because different modes can share the same OMOP value concept ID.

## Feature extraction - pulmonary

#### Feature sputum_amount

In [ ]:
#Sputum amount:the latest sputum amount within 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    o.value_source_value AS sputum_amount

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.observation` o
    ON c.visitId = o.visit_occurrence_id

WHERE o.provider_id IS NOT NULL
    AND o.observation_concept_id = 4089228
    AND o.observation_source_value = 'Hoeveelheid Sputum'
    AND o.observation_datetime <= c.extubationTime
    AND o.observation_datetime >= TIMESTAMP_SUB(
        c.extubationTime, INTERVAL 24 HOUR
    )

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY o.observation_datetime DESC
) = 1
"""

sputum_amount = client.query(sql).to_dataframe()

display(sputum_amount.head(10))


In [ ]:
#Add the feature to the cohort, using left merge to keep all extubation attempts, also if there is no sputum amount recorderd
features = goldStandard.merge(
    sputum_amount,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
)

features = features.drop(columns=["visitId"])

display(features[[
    "visit_occurrence_id",
    "attemptNumber",
    "sputum_amount"
]].head(10))


#Check feature
print("Total extubation attempts:", len(features))

print("\nSputum amount values:")
print(features["sputum_amount"].value_counts(dropna=False))

print("\nCoverage:")
print(features["sputum_amount"].notna().mean())


#### Feature sputum_colour

In [ ]:
#Sputum_color: Get the latest sputum color within 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    o.value_source_value AS sputum_color

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.observation` o
    ON c.visitId = o.visit_occurrence_id

WHERE o.provider_id IS NOT NULL
    AND o.observation_concept_id = 4089228
    AND o.observation_source_value = 'Kleur Sputum'
    AND o.observation_datetime <= c.extubationTime
    AND o.observation_datetime >= TIMESTAMP_SUB(
        c.extubationTime, INTERVAL 24 HOUR
    )

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY o.observation_datetime DESC
) = 1
"""

sputum_color = client.query(sql).to_dataframe()

# Add the feature to our existing dataset
features = features.merge(
    sputum_color,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

# Check the result
print("Total extubation attempts:", len(features))
print("\nSputum color values:")
print(features["sputum_color"].value_counts(dropna=False))
print("\nCoverage:", features["sputum_color"].notna().mean())


#### Feature sputum_consistency

In [ ]:

# Feature: sputum_consistency
# Get the latest sputum consistency within 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.value_source_value AS sputum_consistency

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    ON c.visitId = m.visit_occurrence_id

WHERE m.provider_id IS NOT NULL
    AND m.measurement_concept_id = 4080582
    AND m.measurement_source_value = 'Aspect Sputum'
    AND m.measurement_datetime <= c.extubationTime
    AND m.measurement_datetime >= TIMESTAMP_SUB(
        c.extubationTime, INTERVAL 24 HOUR
    )

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC
) = 1
"""

sputum_consistency = client.query(sql).to_dataframe()

# Add to our feature dataset
features = features.merge(
    sputum_consistency,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

# Check the feature
print("Total extubation attempts:", len(features))
print("\nSputum consistency values:")
print(features["sputum_consistency"].value_counts(dropna=False))
print("\nCoverage:", features["sputum_consistency"].notna().mean())


#### Feature airway_suctioning_count_24h

In [ ]:
#airway_suctioning_count_24h: the number of recorded suctioning procedures in 24 hours before extubation summed

sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    SUM(p.quantity) AS airway_suctioning_count_24h

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.procedure_occurrence` p
    ON c.visitId = p.visit_occurrence_id

WHERE p.provider_id IS NOT NULL
    AND p.procedure_concept_id = 4239191
    AND p.procedure_source_value = 'Aantal Bronchiaaltoilet'
    AND p.quantity IS NOT NULL
    AND p.procedure_datetime <= c.extubationTime
    AND p.procedure_datetime >= TIMESTAMP_SUB(
        c.extubationTime, INTERVAL 24 HOUR
    )

GROUP BY c.visitId, c.attemptNumber
"""

airway_suctioning = client.query(sql).to_dataframe()

# Add to our feature dataset
features = features.merge(
    airway_suctioning,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

# Check the feature
print("Total extubation attempts:", len(features))
print("\nAirway suctioning values:")
print(features["airway_suctioning_count_24h"].describe())

print("\nMissing values:",
      features["airway_suctioning_count_24h"].isna().sum())

print("\nCoverage:",
      features["airway_suctioning_count_24h"].notna().mean())


In [ ]:
display(features[[
    "visit_occurrence_id",
    "attemptNumber",
    "sputum_amount",
    "sputum_color",
    "sputum_consistency",
    "airway_suctioning_count_24h"
]].head(10))

### Feature pneumonia 

In [ ]:
#pneumonia

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    1 AS pneumonia

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` d
    ON c.visitId = d.visit_occurrence_id

WHERE d.provider_id IS NOT NULL
    AND d.condition_concept_id IN ({", ".join(map(str, pneumonia_ids))})
    AND COALESCE(
        d.condition_start_datetime,
        TIMESTAMP(d.condition_start_date)
    ) <= c.extubationTime
"""

pneumonia = client.query(sql).to_dataframe()

features = features.merge(
    pneumonia,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

features["pneumonia"] = features["pneumonia"].fillna(0).astype(int)

print(features["pneumonia"].value_counts())


### Feature COPD related diagnoses
Can also be before 24 hours of extubation because it is a chronic disease

In [ ]:
#Feature: copd_recorded: heck if COPD was recorded at any time before extubation

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    1 AS copd_recorded

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` d
    ON c.visitId = d.visit_occurrence_id

WHERE d.provider_id IS NOT NULL
    AND d.condition_concept_id = 257004
    AND COALESCE(
        d.condition_start_datetime,
        TIMESTAMP(d.condition_start_date)
    ) <= c.extubationTime
"""

copd = client.query(sql).to_dataframe()

features = features.merge(
    copd,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

features["copd_recorded"] = features["copd_recorded"].fillna(0).astype(int)

print(features["copd_recorded"].value_counts())


### Feature ventilation_mode

In [ ]:
#Feature: ventilation_mode
# Get the latest ventilation mode within 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.value_source_value AS ventilation_mode

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m
    ON c.visitId = m.visit_occurrence_id

WHERE m.provider_id IS NOT NULL
    AND m.measurement_concept_id = 3004921
    AND m.value_source_value IS NOT NULL
    AND m.measurement_datetime <= c.extubationTime
    AND m.measurement_datetime >= TIMESTAMP_SUB(
        c.extubationTime, INTERVAL 24 HOUR
    )

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

ventilation_mode = client.query(sql).to_dataframe()

# Add to our feature dataset
features = features.merge(
    ventilation_mode,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

# Check the feature
print("Total extubation attempts:", len(features))
print("\nVentilation mode values:")
print(features["ventilation_mode"].value_counts(dropna=False))
print("\nCoverage:", features["ventilation_mode"].notna().mean())


#### Numerical features pulmonary
Start with latest measured feature within 24 hours of extubation. Later we can do mean, max. min trends etc

In [ ]:
# Concept IDs for the numerical pulmonary variables
numeric_variables = {
    "blood_po2": [3027315],
    "fio2": [3024882],
    "respiratory_rate": [3024171, 3007646],
    "tidal_volume": [21490789, 3015016],
    "tidal_volume_inspired": [3016166]
}

# One audit for all five variables
all_ids = [id for ids in numeric_variables.values() for id in ids]

sql = f"""
SELECT
    measurement_concept_id,
    measurement_source_value,
    unit_source_value,
    unit_concept_id,
    COUNT(*) AS n_records,
    ROUND(MIN(value_as_number), 2) AS min_value,
    ROUND(AVG(value_as_number), 2) AS mean_value,
    ROUND(MAX(value_as_number), 2) AS max_value
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
WHERE provider_id IS NOT NULL
  AND measurement_concept_id IN ({', '.join(map(str, all_ids))})
  AND value_as_number IS NOT NULL
GROUP BY 1, 2, 3, 4
ORDER BY measurement_concept_id, n_records DESC
"""
measurement_audit = client.query(sql).to_dataframe()
display(measurement_audit)


#### observations
- Ademfrequentie Monitor we see outliers: mean= 20.760, max = 3015.0. 
- tidal volume variable set to ml
- fio2 looks like a percentage and not a fraction
- blood_po2 has 2 units mmHg and kPa --> make ut all mmHg


In [ ]:

#Check the distribution of the numerical measurements
sql = f"""
SELECT
    measurement_concept_id,
    measurement_source_value,
    unit_source_value,
    COUNT(*) AS n_records,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(5)], 2) AS p05,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(50)], 2) AS p50,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(95)], 2) AS p95

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

WHERE provider_id IS NOT NULL
    AND measurement_concept_id IN (
        3027315, 3024882, 3024171, 3007646,
        21490789, 3015016, 3016166
    )
    AND value_as_number IS NOT NULL

GROUP BY 1, 2, 3
ORDER BY measurement_concept_id, n_records DESC
"""

measurement_distribution = client.query(sql).to_dataframe()
display(measurement_distribution)


### observations
Some outliers so we have to make a filter for this. And set everything to the same unit. 

In [ ]:

# Numerical pulmonary features: use the ;atest valid measurement, 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,

        CASE
            WHEN measurement_concept_id IN (3024171, 3007646)
                THEN 'respiratory_rate'
            WHEN measurement_concept_id IN (3015016, 21490789)
                THEN 'tidal_volume'
            WHEN measurement_concept_id = 3016166
                THEN 'tidal_volume_inspired'
            WHEN measurement_concept_id = 3024882
                THEN 'fio2'
            WHEN measurement_concept_id = 3027315
                THEN 'blood_po2'
        END AS feature,

        CASE
            WHEN measurement_concept_id = 21490789
                 AND LOWER(unit_source_value) = 'l'
                THEN value_as_number * 1000
            WHEN measurement_concept_id = 3024882
                THEN value_as_number / 100
            WHEN measurement_concept_id = 3027315
                 AND LOWER(unit_source_value) = 'kpa'
                THEN value_as_number * 7.50062
            ELSE value_as_number
        END AS value

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
    WHERE provider_id IS NOT NULL
      AND measurement_concept_id IN (
          3024171, 3007646, 3015016, 21490789,
          3016166, 3024882, 3027315
      )
      AND value_as_number IS NOT NULL
      AND (
          measurement_concept_id NOT IN (3015016, 21490789, 3016166, 3027315)
          OR (
              measurement_concept_id IN (3015016, 3016166)
              AND LOWER(unit_source_value) = 'ml'
          )
          OR (
              measurement_concept_id = 21490789
              AND LOWER(unit_source_value) IN ('ml', 'l')
          )
          OR (
              measurement_concept_id = 3027315
              AND LOWER(unit_source_value) IN ('mmhg', 'kpa')
          )
      )
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE (feature = 'respiratory_rate' AND value BETWEEN 4 AND 80)
       OR (feature = 'tidal_volume' AND value BETWEEN 50 AND 2000)
       OR (feature = 'tidal_volume_inspired' AND value BETWEEN 50 AND 2000)
       OR (feature = 'fio2' AND value BETWEEN 0.21 AND 1)
       OR (feature = 'blood_po2' AND value BETWEEN 20 AND 600)
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.feature,
    m.value

FROM cohort c
JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber, m.feature
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

numeric_results = client.query(sql).to_dataframe()

display(numeric_results.head(10))
#every row is a feature value for a extubation attempt


In [ ]:
#num values next to each other in columns with the use of pivot()

numeric_features = numeric_results.pivot(
    index=["visitId", "attemptNumber"],
    columns="feature",
    values="value"
).reset_index()

# Add to existing feature dataset
features = features.merge(
    numeric_features,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

print("Total extubation attempts:", len(features))
display(features.head())


In [ ]:
print(numeric_features.columns.tolist())
print(features.columns.tolist())

In [ ]:

# Check the numerical features
numeric_columns = [
    "respiratory_rate",
    "tidal_volume",
    "tidal_volume_inspired",
    "fio2",
    "blood_po2",
    "airway_suctioning_count_24h"
]

# Convert values to floats
features[numeric_columns] = features[numeric_columns].astype(float)

print("Coverage:")
print(features[numeric_columns].notna().mean())

print("\nValue distributions:")
display(features[numeric_columns].describe())


# Feature engineering - Sepsis

In [ ]:
# Check numerical sepsis measurements: units, values and distributions

sepsis_numeric_ids = [
    21490588, 21490870, 21490586, 3022060,
    3006322, 3025163, 3025085,  # Temperature
    3010813,                      # Leukocytes
    3047181, 3014111,             # Lactate
    3007461,                      # Platelets
    3006140, 40757494,            # Bilirubin
    3020564,                      # Creatinine
    21490852                      # MAP
]

sql = f"""
SELECT
    measurement_concept_id,
    measurement_source_value,
    unit_source_value,
    COUNT(*) AS n_records,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(5)], 2) AS p05,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(50)], 2) AS p50,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(95)], 2) AS p95

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

WHERE provider_id IS NOT NULL
    AND measurement_concept_id IN ({", ".join(map(str, sepsis_numeric_ids))})
    AND value_as_number IS NOT NULL

GROUP BY 1, 2, 3
ORDER BY measurement_concept_id, n_records DESC
"""

sepsis_measurements = client.query(sql).to_dataframe()
display(sepsis_measurements)


### Interpretation of numerical features
lab values are within the same unit

only check differences for temperature measured at different locations. Leave:3006322: temp_oraal out of the feature because the median is much lower compared to the other temp measurements and only 546 registrations. 

### Make numerical features for sepsis

In [ ]:
# Numerical sepsis features: take the latest valid measurement within 24 hours before extubation

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        value_as_number AS value,

        CASE
            WHEN measurement_concept_id IN (
                21490588, 21490870, 21490586,
                3022060, 3025163, 3025085
            ) THEN 'temperature'
            WHEN measurement_concept_id = 3010813 THEN 'leukocytes'
            WHEN measurement_concept_id IN (3047181, 3014111) THEN 'lactate'
            WHEN measurement_concept_id = 3007461 THEN 'platelets'
            WHEN measurement_concept_id IN (3006140, 40757494) THEN 'bilirubin'
            WHEN measurement_concept_id = 3020564 THEN 'creatinine'
            WHEN measurement_concept_id = 21490852 THEN 'map'
        END AS feature

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`
    WHERE provider_id IS NOT NULL
      AND measurement_concept_id IN (
          21490588, 21490870, 21490586, 3022060,
          3025163, 3025085, 3010813, 3047181,
          3014111, 3007461, 3006140, 40757494,
          3020564, 21490852
      )
      AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE (feature = 'temperature' AND value BETWEEN 30 AND 42)
       OR (feature = 'leukocytes' AND value BETWEEN 0.1 AND 200)
       OR (feature = 'lactate' AND value BETWEEN 0.1 AND 30)
       OR (feature = 'platelets' AND value BETWEEN 1 AND 1500)
       OR (feature = 'bilirubin' AND value BETWEEN 1 AND 1000)
       OR (feature = 'creatinine' AND value BETWEEN 5 AND 2000)
       OR (feature = 'map' AND value BETWEEN 30 AND 180)
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.feature,
    m.value

FROM cohort c
JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber, m.feature
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

sepsis_numeric_results = client.query(sql).to_dataframe()

display(sepsis_numeric_results.head(10))


In [ ]:
# Convert numerical sepsis features into columns
sepsis_numeric_features = sepsis_numeric_results.pivot(
    index=["visitId", "attemptNumber"],
    columns="feature",
    values="value"
).reset_index()

# Add to our feature dataset
features = features.merge(
    sepsis_numeric_features,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

print("Total extubation attempts:", len(features))
display(features.head())


In [ ]:

# Check numerical sepsis features

sepsis_numeric_columns = [
    "temperature",
    "leukocytes",
    "lactate",
    "platelets",
    "bilirubin",
    "creatinine",
    "map"
]

# Convert values to floats
features[sepsis_numeric_columns] = (
    features[sepsis_numeric_columns].astype(float)
)

print("Coverage:")
print(features[sepsis_numeric_columns].notna().mean())

print("\nValue distributions:")
display(features[sepsis_numeric_columns].describe())


### sepsis diagnosis variable
look anywere before extubation not limited to 24 hours 

In [ ]:
# Feature: sepsis_diagnosis was sepsis diagnosed before extubation 

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    1 AS sepsis_diagnosis

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` d
    ON c.visitId = d.visit_occurrence_id

WHERE d.provider_id IS NOT NULL
    AND d.condition_concept_id = 132797
    AND COALESCE(
        d.condition_start_datetime,
        TIMESTAMP(d.condition_start_date)
    ) <= c.extubationTime
"""

sepsis_diagnosis = client.query(sql).to_dataframe()

features = features.merge(
    sepsis_diagnosis,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

features["sepsis_diagnosis"] = (
    features["sepsis_diagnosis"].fillna(0).astype(int)
)

print(features["sepsis_diagnosis"].value_counts())


### Sepsis suspected variable
Not limited to 24 hours
id: 44808681 contains the answers ja, nee and onbekend
Ja = 1
Nee = 0 
Onbekend or No registration is Nan

In [ ]:
# Feature: suspected_sepsis

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    o.value_source_value AS sepsis_status

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.observation` o
    ON c.visitId = o.visit_occurrence_id

WHERE o.provider_id IS NOT NULL
    AND o.observation_concept_id = 44808681
    AND o.observation_datetime <= c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY o.observation_datetime DESC, o.observation_id DESC
) = 1
"""

suspected_sepsis = client.query(sql).to_dataframe()

# Convert Ja/Nee/Onbekend to 1/0/NaN
suspected_sepsis["suspected_sepsis"] = (
    suspected_sepsis["sepsis_status"].map({
        "Ja": 1,
        "Nee": 0
    })
)

features = features.merge(
    suspected_sepsis[["visitId", "attemptNumber", "suspected_sepsis"]],
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

print(features["suspected_sepsis"].value_counts(dropna=False))


### Inspection of Antibiotics and vasoactive support variables
Noticed that for medication registrations there was only provider_id IS NULL, otherwise we didnt find any medication

In [ ]:

# Check medication registrations

medication_variables = {
    "antibiotics": [43295826, 36881808, 40928582, 21088706, 40737711, 40077149, 40731671],
    "vasoactive_support": [2907531, 40072394, 40074086, 21088391, 36411287, 19076867, 40038600]
}

medication_ids = medication_variables["antibiotics"] + medication_variables["vasoactive_support"]

sql = f"""
SELECT
    drug_concept_id,
    drug_source_value,
    COUNT(*) AS n_records,
    COUNTIF(drug_exposure_start_datetime IS NULL) AS missing_start,
    COUNTIF(drug_exposure_end_datetime IS NULL) AS missing_end
FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure`
WHERE provider_id IS NULL
    AND drug_concept_id IN ({", ".join(map(str, medication_ids))})
GROUP BY 1, 2
ORDER BY n_records DESC
"""

display(client.query(sql).to_dataframe())


### feature antibiotics_24H

In [ ]:
#antibiotics_24h: antibiotics started 24H for extubation

antibiotic_ids = ", ".join(map(str, medication_variables["antibiotics"]))

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    1 AS antibiotics_24h

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` d
    ON c.visitId = d.visit_occurrence_id

WHERE d.provider_id IS NULL
    AND d.drug_concept_id IN ({antibiotic_ids})
    AND d.drug_exposure_start_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
"""

antibiotics = client.query(sql).to_dataframe()

features = features.merge(
    antibiotics,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

features["antibiotics_24h"] = features["antibiotics_24h"].fillna(0).astype(int)
print(features["antibiotics_24h"].value_counts())

### feature vasoactive_support_24h
Vasoactive medication registrations often cover longer intervals, suggesting infusion periods. Therefore, vasoactive_support_24h is defined as the presence of at least one recorded vasoactive exposure overlapping the 24-hour window before extubation.

In [ ]:
# Feature: vasoactive_support_24h

vasoactive_ids = ", ".join(map(str, medication_variables["vasoactive_support"]))

sql = f"""
WITH cohort AS (
    SELECT * FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    1 AS vasoactive_support_24h

FROM cohort c
JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.drug_exposure` d
    ON c.visitId = d.visit_occurrence_id

WHERE d.provider_id IS NULL
    AND d.drug_concept_id IN ({vasoactive_ids})
    AND d.drug_exposure_start_datetime <= c.extubationTime
    AND d.drug_exposure_end_datetime >=
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
"""

vasoactive_support = client.query(sql).to_dataframe()

features = features.merge(
    vasoactive_support,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
).drop(columns=["visitId"])

features["vasoactive_support_24h"] = features["vasoactive_support_24h"].fillna(0).astype(int)
print(features["vasoactive_support_24h"].value_counts())

### Check sepsis features

In [ ]:
#sepsis features check

sepsis_features = [
    "temperature",
    "leukocytes",
    "lactate",
    "platelets",
    "bilirubin",
    "creatinine",
    "map",
    "sepsis_diagnosis",
    "suspected_sepsis",
    "antibiotics_24h",
    "vasoactive_support_24h"
]

print("Extubation attempts:", len(features))
print("Duplicates:",
      features.duplicated(["visit_occurrence_id", "attemptNumber"]).sum())

print("\nSepsis feature coverage:")
print(features[sepsis_features].notna().mean())

print("\nFirst rows:")
display(features[sepsis_features + ["failure"]].head(10))


In [ ]:

# Check binary sepsis features
binary_columns = [
    "sepsis_diagnosis",
    "antibiotics_24h",
    "vasoactive_support_24h"
]

for column in binary_columns:
    print(f"\n{column}:")
    print(features[column].value_counts())
    print("Percentage positive:",
          round(features[column].mean() * 100, 2), "%")


# Feature engineering - Cardiac


In [ ]:
cardiac_numeric_variables = {
    "heart_rate": [3027018, 21490872],
    "invasive_sbp": [21490853],
    "invasive_dbp": [21490851],
    "cvp": [21490675],
    "lactate": [3047181, 3014111],
    "glucose": [3020491],
    "hemoglobin": [40762351],
    "ck_mb": [3005785],
    "tropine": [3019800, 3048529],
    "potassium": [3023103, 3005456],
    "urea": [43534077],
    "creatinine": [3020564]
}

cardiac_numeric_ids = [
    id
    for ids in cardiac_numeric_variables.values()
    for id in ids
]

Variable Heart-Rate: 

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3027018, 21490872)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(
        APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2
    ) AS p01,

    ROUND(
        APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2
    ) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

heart_rate_distribution = client.query(sql).to_dataframe()

display(heart_rate_distribution)

 We see 3 different source values: IABP HF, PiCCO HF Hartfrequentie and hartfrequent
* both the same measurement /min 
* IABP has the most measurements
* We choose to combine both measures, and use clipping fot the first and 99th percentile
* No extreme variables in this cohort
* We only combine the variables to the feature: heart_rate

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS heart_rate

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3027018, 21490872)
        AND measurement_source_value IN (
            'IABP HF',
            'PiCCO HF Hartfrequentie',
            'Hartfrequentie'
        )
        AND value_as_number IS NOT NULL
        AND unit_source_value = '/min'
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.heart_rate

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

heart_rate = client.query(sql).to_dataframe()

print("Total heart_rate records:", len(heart_rate))

Invasive SBP:

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (21490853)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(
        APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2
    ) AS p01,

    ROUND(
        APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2
    ) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

invasive_sbp = client.query(sql).to_dataframe()

display(invasive_sbp)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS invasive_sbp

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 21490853
        AND value_as_number IS NOT NULL
        AND unit_source_value = 'mmHg'
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.invasive_sbp

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

invasive_sbp_measurements = client.query(sql).to_dataframe()

In [ ]:
sbp_outliers = invasive_sbp_measurements[
    (invasive_sbp_measurements["invasive_sbp"] < 40) |
    (invasive_sbp_measurements["invasive_sbp"] > 250)
]
print("Total SBP measurements:", len(invasive_sbp_measurements))
print("Number of outliers:", len(sbp_outliers))
print("Number of measurements within range:", len(invasive_sbp_measurements) - len(sbp_outliers))

* a range of 40-250 is clinical correct and remains enought data
* SBP will containe the combined measurements in a range of 40-250
* variable: invasive_sbp

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS invasive_sbp

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 21490853
        AND value_as_number IS NOT NULL
        AND unit_source_value = 'mmHg'
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE invasive_sbp BETWEEN 40 AND 250
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.invasive_sbp

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

invasive_sbp = client.query(sql).to_dataframe()

# Invasive_dpb

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (21490851)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

invasive_dbp = client.query(sql).to_dataframe()

display(invasive_dbp)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS invasive_dbp

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 21490851
        AND value_as_number IS NOT NULL
        AND unit_source_value = 'mmHg'
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.invasive_dbp

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

invasive_dbp_measurements = client.query(sql).to_dataframe()

In [ ]:
dbp_outliers = invasive_dbp_measurements[
    (invasive_dbp_measurements["invasive_dbp"] < 20) |
    (invasive_dbp_measurements["invasive_dbp"] > 150)
]
print("Total DBP measurements:", len(invasive_dbp_measurements))
print("Number of outliers:", len(dbp_outliers))
print("Number of measurements within range:", len(invasive_dbp_measurements) - len(dbp_outliers))

* a range of 20-150 is clinical correct and remains enought data
* DBP will containe the combined measurements in a range of 420-150
* variable: invasive_dbp

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS invasive_dbp

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 21490851
        AND value_as_number IS NOT NULL
        AND unit_source_value = 'mmHg'
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE invasive_dbp BETWEEN 20 AND 150
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.invasive_dbp

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

invasive_dbp = client.query(sql).to_dataframe()

Central Venous Pressure 

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (21490675)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

cvp = client.query(sql).to_dataframe()

display(cvp)

No extreme values. 
* variable = cvp

lactate

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3047181, 3014111)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

lactate = client.query(sql).to_dataframe()

display(lactate)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS lactate

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3047181, 3014111)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.lactate

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

lactate_measurements = client.query(sql).to_dataframe()

In [ ]:
lactate_outliers = lactate_measurements[
    (lactate_measurements["lactate"] < 0.3) |
    (lactate_measurements["lactate"] > 5)
]
print("Total latcate measurements:", len(lactate_measurements))
print("Number of outliers:", len(lactate_outliers))
print("Number of measurements within range:", len(lactate_measurements) - len(lactate_outliers))

* a range of 0.3-5 is clinical correct and remains enought data
variable: lactate


In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS lactate

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3047181, 3014111)
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE lactate BETWEEN 0.3 AND 5
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.lactate

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

lactate = client.query(sql).to_dataframe()

Atriumfibrilatie


In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

conditions AS (
    SELECT
        visit_occurrence_id,
        condition_concept_id,
        condition_source_value,
        condition_start_datetime

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence`

    WHERE provider_id IS NOT NULL
        AND condition_concept_id IN (313217)
)

SELECT
    c.visitId,
    c.attemptNumber,
    co.condition_concept_id,
    co.condition_source_value,
    COUNT(*) AS n_records

FROM cohort c

JOIN conditions co
    ON c.visitId = co.visit_occurrence_id

WHERE co.condition_start_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3, 4

ORDER BY condition_concept_id, n_records DESC
"""

atrium_fibrilatie = client.query(sql).to_dataframe()

display(atrium_fibrilatie)

* We want to make a boolean feature of atrium fibrilation
* added at the end of the notebook

In [ ]:

atrium_fibrilatie["atrial_fibrillation"] = 1
afib_feature = (
    atrium_fibrilatie[
        ["visitId", "attemptNumber", "atrial_fibrillation"]
    ]
    .drop_duplicates()
)

In [ ]:
goldStandard = goldStandard.merge(
    afib_feature,
    left_on=["visit_occurrence_id", "attemptNumber"],
    right_on=["visitId", "attemptNumber"],
    how="left"
)

goldStandard["atrial_fibrillation"] = (
    goldStandard["atrial_fibrillation"]
    .fillna(0)
    .astype(int)
)

# Remove duplicate visit identifier
goldStandard = goldStandard.drop(columns=["visitId"])

In [ ]:
print(goldStandard["atrial_fibrillation"].value_counts())

display(goldStandard.head())

Glucose


In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3020491)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

glucose = client.query(sql).to_dataframe()

display(glucose)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS glucose

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3020491)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.glucose

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

glucose_measurements = client.query(sql).to_dataframe()

In [ ]:
glucose_outliers = glucose_measurements[
    (glucose_measurements["glucose"] < 1.5) |
    (glucose_measurements["glucose"] > 30)
]
print("Total latcate measurements:", len(glucose_measurements))
print("Number of outliers:", len(glucose_outliers))
print("Number of measurements within range:", len(glucose_measurements) - len(glucose_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS glucose

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3020491
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE glucose BETWEEN 1.5 AND 30
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.glucose

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

glucose = client.query(sql).to_dataframe()

Hemoglobine

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (40762351)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

hemoglobine = client.query(sql).to_dataframe()

display(hemoglobine)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS hemoglobine

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (40762351)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.hemoglobine

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

hemoglobine_measurements = client.query(sql).to_dataframe()

In [ ]:
glucose_outliers = glucose_measurements[
    (glucose_measurements["glucose"] < 2) |
    (glucose_measurements["glucose"] > 15)
]
print("Total latcate measurements:", len(glucose_measurements))
print("Number of outliers:", len(glucose_outliers))
print("Number of measurements within range:", len(glucose_measurements) - len(glucose_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS hemoglobine

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 40762351
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE hemoglobine BETWEEN 2 AND 15
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.hemoglobine

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

hemoglobine = client.query(sql).to_dataframe()

CK MB

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3005785)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

ck_mb = client.query(sql).to_dataframe()

display(ck_mb)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS ck_mb

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3005785)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.ck_mb

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

ck_mb_measurements = client.query(sql).to_dataframe()

In [ ]:
ck_mb_outliers = ck_mb_measurements[
    (ck_mb_measurements["ck_mb"] < 0) |
    (ck_mb_measurements["ck_mb"] > 100)
]
print("Total ck_mb measurements:", len(ck_mb_measurements))
print("Number of outliers:", len(ck_mb_outliers))
print("Number of measurements within range:", len(ck_mb_measurements) - len(ck_mb_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS ck_mb

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3005785
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE ck_mb BETWEEN 0 AND 100
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.ck_mb

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

ck_mb = client.query(sql).to_dataframe()

Tropine

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3019800, 3048529)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

tropine = client.query(sql).to_dataframe()

display(tropine)

potassium

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3023103, 3005456)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

potassium = client.query(sql).to_dataframe()

display(potassium)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS kalium

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3023103, 3005456)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.kalium

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

kalium_measurements = client.query(sql).to_dataframe()

In [ ]:
kalium_outliers = kalium_measurements[
    (kalium_measurements["kalium"] < 1) |
    (kalium_measurements["kalium"] > 8)
]
print("Total kalium measurements:", len(kalium_measurements))
print("Number of outliers:", len(kalium_outliers))
print("Number of measurements within range:", len(kalium_measurements) - len(kalium_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS kalium

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3023103, 3005456)
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE kalium BETWEEN 1 AND 8
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.kalium

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

kalium = client.query(sql).to_dataframe()

Urea

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (43534077)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

Urea = client.query(sql).to_dataframe()

display(Urea)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS ureum

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (43534077)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.ureum

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

ureum_measurements = client.query(sql).to_dataframe()

In [ ]:
ureum_outliers = ureum_measurements[
    (ureum_measurements["ureum"] < 0.5) |
    (ureum_measurements["ureum"] > 50)
]
print("Total latcate measurements:", len(ureum_measurements))
print("Number of outliers:", len(ureum_outliers))
print("Number of measurements within range:", len(ureum_measurements) - len(ureum_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS ureum

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 43534077
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE ureum BETWEEN 0.5 AND 50
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.ureum

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

ureum = client.query(sql).to_dataframe()

creatinine

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3020564)
        AND value_as_number IS NOT NULL
)

SELECT
    m.measurement_concept_id,
    m.measurement_source_value,
    m.unit_source_value,
    COUNT(*) AS n_records,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(1)], 2) AS p01,

    ROUND(APPROX_QUANTILES(m.value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(m.value_as_number), 2) AS min_value,
    ROUND(MAX(m.value_as_number), 2) AS max_value

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

creatinine = client.query(sql).to_dataframe()

display(creatinine)

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS creatinine

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3020564)
        AND value_as_number IS NOT NULL
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.measurement_datetime,
    m.measurement_id,
    m.measurement_source_value,
    m.creatinine

FROM cohort c

JOIN measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime
"""

creatinine_measurements = client.query(sql).to_dataframe()

In [ ]:
creatinine_outliers = creatinine_measurements[
    (creatinine_measurements["creatinine"] < 20) |
    (creatinine_measurements["creatinine"] > 500)
]
print("Total latcate measurements:", len(creatinine_measurements))
print("Number of outliers:", len(creatinine_outliers))
print("Number of measurements within range:", len(creatinine_measurements) - len(creatinine_outliers))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_source_value,
        value_as_number AS creatinine

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3020564
        AND value_as_number IS NOT NULL
),

valid_measurements AS (
    SELECT *
    FROM measurements
    WHERE creatinine BETWEEN 20 AND 500
)

SELECT
    c.visitId,
    c.attemptNumber,
    m.creatinine

FROM cohort c

JOIN valid_measurements m
    ON c.visitId = m.visit_occurrence_id

WHERE m.measurement_datetime BETWEEN
    TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
    AND c.extubationTime

QUALIFY ROW_NUMBER() OVER (
    PARTITION BY c.visitId, c.attemptNumber
    ORDER BY m.measurement_datetime DESC, m.measurement_id DESC
) = 1
"""

creatinine = client.query(sql).to_dataframe()

In [ ]:
goldStandard = goldStandard[["person_id", "visit_occurrence_id", "startTime",
                             "extubationTime", "attemptNumber", "failure"]]

In [ ]:
def add_feature(features, df, col):
    features = features.drop(columns=col, errors="ignore") 
    out = features.merge(df[["visitId", "attemptNumber", col]],
                         left_on=["visit_occurrence_id", "attemptNumber"],
                         right_on=["visitId", "attemptNumber"], how="left").drop(columns="visitId")
    assert len(out) == len(features)
    return out

In [ ]:
def latest_value(concept_ids, name, low=None, high=None):
    ids = ", ".join(map(str, concept_ids))
    rng = "".join([f" AND m.value_as_number >= {low}" if low is not None else "",
                   f" AND m.value_as_number <= {high}" if high is not None else ""])
    sql = f"""
    WITH cohort AS (SELECT * FROM UNNEST([{cohortRows}]))
    SELECT c.visitId, c.attemptNumber, m.value_as_number AS {name}
    FROM cohort c
    JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement` m ON c.visitId = m.visit_occurrence_id
    WHERE m.provider_id IS NOT NULL AND m.measurement_concept_id IN ({ids})
      AND m.value_as_number IS NOT NULL{rng}
      AND m.measurement_datetime BETWEEN TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR) AND c.extubationTime
    QUALIFY ROW_NUMBER() OVER (PARTITION BY c.visitId, c.attemptNumber
                               ORDER BY m.measurement_datetime DESC, m.measurement_id DESC) = 1
    """
    df = client.query(sql).to_dataframe()
    df[name] = df[name].astype(float)
    return df

In [ ]:
invasive_sbp = latest_value([21490853], "invasive_sbp", low=40, high=250)
cvp = latest_value([21490675], "cvp")
tropine = latest_value([3019800], "tropine") 

In [ ]:
invasive_dbp = latest_value(
    [21490851],
    "invasive_dbp",
    low=20,
    high=150
)

In [ ]:
features = goldStandard.copy()

for name, df in {
    "heart_rate": heart_rate,
    "invasive_sbp": invasive_sbp,
    "invasive_dbp": invasive_dbp,
    "cvp": cvp,
    "glucose": glucose,
    "hemoglobine": hemoglobine,
    "ck_mb": ck_mb,
    "tropine": tropine,
    "kalium": kalium,
    "ureum": ureum
}.items():

    features = add_feature(features, df, name)

In [ ]:
print(features.shape)
print(features.columns.tolist())
features.head()

In [ ]:
for name, df in {
    "heart_rate": heart_rate,
    "invasive_sbp": invasive_sbp,
    "invasive_dbp": invasive_dbp,
    "cvp": cvp,
    "glucose": glucose,
    "hemoglobine": hemoglobine,
    "ck_mb": ck_mb,
    "tropine": tropine,
    "kalium": kalium,
    "ureum": ureum
}.items():
    print(f"{name}: {df.columns.tolist()}")

In [ ]:
atrium_fibrilatie["atrial_fibrillation"] = 1
afib_feature = (
    atrium_fibrilatie[
        ["visitId", "attemptNumber", "atrial_fibrillation"]
    ]
    .drop_duplicates()
)

In [ ]:
print("Aantal rijen goldStandard:", len(goldStandard))
print("Aantal rijen afib_feature:", len(afib_feature))

print("\nVoorbeelden goldStandard:")
display(goldStandard[["visit_occurrence_id", "attemptNumber"]].head())

print("\nVoorbeelden afib_feature:")
display(afib_feature.head())


# Feature engineering - neuro

In [ ]:
neurological_variables = {
    "gcs_motor": [3008223],
    "gcs_verbal": [3009094],
    "gcs_eye": [3016335],
    "rass": [36684829],
    "eye_pupillary ": [21491763, 21490963],
    "eye_diameter ": [3026611, 3010321],
    "epileptic_seizure": [4197485]
}

In [ ]:
concept_mapping = {
    concept_id: variable
    for variable, concept_ids in neurological_variables.items()
    for concept_id in concept_ids
}

all_concept_ids = list(concept_mapping.keys())

concept_ids_sql = ", ".join(map(str, all_concept_ids))

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_concept_id,
        measurement_source_value,
        unit_source_value,
        measurement_datetime,
        value_as_number

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN ({concept_ids_sql})
),

cohort_measurements AS (
    SELECT
        m.*
    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
)

SELECT
    measurement_concept_id,
    measurement_source_value,
    unit_source_value,

    COUNT(*) AS n_records,
    COUNT(DISTINCT visit_occurrence_id) AS n_visits,

    ROUND(AVG(value_as_number), 2) AS mean_value,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(1)], 2) AS p01,
    ROUND(APPROX_QUANTILES(value_as_number, 100)[OFFSET(99)], 2) AS p99,

    ROUND(MIN(value_as_number), 2) AS min_value,
    ROUND(MAX(value_as_number), 2) AS max_value

FROM cohort_measurements

GROUP BY 1, 2, 3

ORDER BY measurement_concept_id, n_records DESC
"""

neurological_summary = client.query(sql).to_dataframe()

neurological_summary["variable"] = (
    neurological_summary["measurement_concept_id"].map(concept_mapping)
)

display(neurological_summary)

I used this query to look up the different variables one by one

In [ ]:
sql = f"""
SELECT
    condition_concept_id,
    condition_source_value,
    COUNT(*) AS n_records

FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence`

WHERE provider_id IS NOT NULL
    AND condition_concept_id IN (4197485)

GROUP BY 1, 2

ORDER BY
    condition_concept_id,
    n_records DESC
"""

neurological_values = client.query(sql).to_dataframe()

neurological_values["variable"] = (
    neurological_values["condition_concept_id"].map(concept_mapping)
)

display(neurological_values)

making numeric values of the gcl_motor

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3008223
        AND value_as_concept_id IS NOT NULL
),

gcs_motor AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.value_as_concept_id = 45880468 THEN 6
            WHEN m.value_as_concept_id = 45880467 THEN 5
            WHEN m.value_as_concept_id = 45882047 THEN 4
            WHEN m.value_as_concept_id = 45879885 THEN 3
            WHEN m.value_as_concept_id = 45878993 THEN 2
            WHEN m.value_as_concept_id = 45878992 THEN 1
        END AS gcs_motor_score

    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
)

SELECT
    visitId,
    attemptNumber,

    MAX(gcs_motor_score) AS gcs_motor_max,
    MIN(gcs_motor_score) AS gcs_motor_min,

    ARRAY_AGG(
        gcs_motor_score
        ORDER BY measurement_datetime DESC, measurement_id DESC
        LIMIT 1
    )[OFFSET(0)] AS gcs_motor_last,

    COUNT(gcs_motor_score) AS gcs_motor_n_measurements

FROM gcs_motor

WHERE gcs_motor_score IS NOT NULL

GROUP BY
    visitId,
    attemptNumber
"""

gcs_motor_features = client.query(sql).to_dataframe()

display(gcs_motor_features)

also for glc_verbal

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3009094
        AND value_as_concept_id IS NOT NULL
),

gcs_verbal AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.value_as_concept_id = 45877602 THEN 5
            WHEN m.value_as_concept_id = 45883906 THEN 4
            WHEN m.value_as_concept_id = 45877601 THEN 3
            WHEN m.value_as_concept_id = 45883352 THEN 2
            WHEN m.value_as_concept_id = 45877384 THEN 1
            WHEN m.value_as_concept_id = 36311192 THEN NULL
        END AS gcs_verbal_score

    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
)

SELECT
    visitId,
    attemptNumber,

    MAX(gcs_verbal_score) AS gcs_verbal_max,
    MIN(gcs_verbal_score) AS gcs_verbal_min,

    ARRAY_AGG(
        gcs_verbal_score
        ORDER BY measurement_datetime DESC, measurement_id DESC
        LIMIT 1
    )[OFFSET(0)] AS gcs_verbal_last,

    COUNT(gcs_verbal_score) AS gcs_verbal_n_measurements

FROM gcs_verbal

WHERE gcs_verbal_score IS NOT NULL

GROUP BY
    visitId,
    attemptNumber
"""

gcs_verbal_features = client.query(sql).to_dataframe()

display(gcs_verbal_features)

also for gcs_eye

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 3016335
        AND value_as_concept_id IS NOT NULL
),

gcs_eye AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.value_as_concept_id = 45880466 THEN 4
            WHEN m.value_as_concept_id IN (45876254, 45880465) THEN 3
            WHEN m.value_as_concept_id IN (45884209, 45883351) THEN 2
            WHEN m.value_as_concept_id = 45877537 THEN 1
        END AS gcs_eye_score

    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
)

SELECT
    visitId,
    attemptNumber,

    MAX(gcs_eye_score) AS gcs_eye_max,
    MIN(gcs_eye_score) AS gcs_eye_min,

    ARRAY_AGG(
        gcs_eye_score
        ORDER BY measurement_datetime DESC, measurement_id DESC
        LIMIT 1
    )[OFFSET(0)] AS gcs_eye_last,

    COUNT(gcs_eye_score) AS gcs_eye_n_measurements

FROM gcs_eye

WHERE gcs_eye_score IS NOT NULL

GROUP BY
    visitId,
    attemptNumber
"""

gcs_eye_features = client.query(sql).to_dataframe()

display(gcs_eye_features)

In [ ]:
df_gcs["gcs_total"] = df_gcs[
    ["gcs_motor_last", "gcs_verbal_last_observable", "gcs_eye_last"]
].sum(axis=1, min_count=3)

RASS score

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id = 36684829
        AND value_as_concept_id IS NOT NULL
),

rass AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.value_as_concept_id = 45880051 THEN -5
            WHEN m.value_as_concept_id = 45877871 THEN -1
            WHEN m.value_as_concept_id = 1990543  THEN -3
            WHEN m.value_as_concept_id = 45879434 THEN -4
            WHEN m.value_as_concept_id = 46237183 THEN -2
            WHEN m.value_as_concept_id = 45884808 THEN 0
            WHEN m.value_as_concept_id = 45881739 THEN 1
            WHEN m.value_as_concept_id = 46237198 THEN 2
            WHEN m.value_as_concept_id = 45877463 THEN 4
        END AS rass_score

    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
)

SELECT
    visitId,
    attemptNumber,

    MAX(rass_score) AS rass_max,
    MIN(rass_score) AS rass_min,

    ARRAY_AGG(
        rass_score
        ORDER BY measurement_datetime DESC, measurement_id DESC
        LIMIT 1
    )[OFFSET(0)] AS rass_last,

    COUNT(rass_score) AS rass_n_measurements

FROM rass

WHERE rass_score IS NOT NULL

GROUP BY
    visitId,
    attemptNumber
"""

rass_features = client.query(sql).to_dataframe()

display(rass_features)

Eye pupilar:

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (21490963, 21491763)
        AND value_as_concept_id IS NOT NULL
),

pupil_reaction AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.measurement_concept_id = 21490963 THEN 'right'
            WHEN m.measurement_concept_id = 21491763 THEN 'left'
        END AS pupil_side,

        CASE
            WHEN m.value_as_concept_id = 21498502 THEN 1
            WHEN m.value_as_concept_id = 21498863 THEN 0
        END AS pupil_reactive

    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
),

pupil_features AS (
    SELECT
        visitId,
        attemptNumber,

        -- Rechter pupil
        ARRAY_AGG(
            IF(pupil_side = 'right', pupil_reactive, NULL)
            IGNORE NULLS
            ORDER BY measurement_datetime DESC, measurement_id DESC
            LIMIT 1
        )[SAFE_OFFSET(0)] AS pupil_right_reactive_last,

        COUNTIF(pupil_side = 'right' AND pupil_reactive IS NOT NULL)
            AS pupil_right_n_measurements,

        -- Linker pupil
        ARRAY_AGG(
            IF(pupil_side = 'left', pupil_reactive, NULL)
            IGNORE NULLS
            ORDER BY measurement_datetime DESC, measurement_id DESC
            LIMIT 1
        )[SAFE_OFFSET(0)] AS pupil_left_reactive_last,

        COUNTIF(pupil_side = 'left' AND pupil_reactive IS NOT NULL)
            AS pupil_left_n_measurements

    FROM pupil_reaction

    GROUP BY
        visitId,
        attemptNumber
)

SELECT
    *,

    CASE
        WHEN pupil_right_reactive_last = 1
         AND pupil_left_reactive_last = 1
        THEN 1
        WHEN pupil_right_reactive_last = 0
          OR pupil_left_reactive_last = 0
        THEN 0
    END AS pupil_both_reactive_last,

    CASE
        WHEN pupil_right_reactive_last = 0
          OR pupil_left_reactive_last = 0
        THEN 1
        WHEN pupil_right_reactive_last = 1
         AND pupil_left_reactive_last = 1
        THEN 0
    END AS pupil_any_unreactive_last

FROM pupil_features
"""

pupil_features = client.query(sql).to_dataframe()

display(pupil_features)

Eye diameter


In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

measurements AS (
    SELECT
        visit_occurrence_id,
        measurement_datetime,
        measurement_id,
        measurement_concept_id,
        value_as_concept_id

    FROM `{DATASET_PROJECT_ID}.{DATASET_ID}.measurement`

    WHERE provider_id IS NOT NULL
        AND measurement_concept_id IN (3010321, 3026611)
        AND value_as_concept_id IS NOT NULL
),

pupil_size AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        m.measurement_datetime,
        m.measurement_id,

        CASE
            WHEN m.measurement_concept_id = 3010321 THEN 'right'
            WHEN m.measurement_concept_id = 3026611 THEN 'left'
        END AS pupil_side,

        CASE
            WHEN m.value_as_concept_id = 45881054 THEN 1
            WHEN m.value_as_concept_id = 45884805 THEN 2
            WHEN m.value_as_concept_id = 45880155 THEN 3
            WHEN m.value_as_concept_id = 45877193 THEN 4
            WHEN m.value_as_concept_id = 45883038 THEN 5
            WHEN m.value_as_concept_id = 45883904 THEN 6
            WHEN m.value_as_concept_id = 45877461 THEN 7
            WHEN m.value_as_concept_id = 45884806 THEN 8
        END AS pupil_size_mm,


    FROM cohort c

    JOIN measurements m
        ON c.visitId = m.visit_occurrence_id

    WHERE m.measurement_datetime BETWEEN
        TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
        AND c.extubationTime
),

pupil_features AS (
    SELECT
        visitId,
        attemptNumber,

        -- Rechter pupil
        ARRAY_AGG(
            IF(pupil_side = 'right', pupil_size_mm, NULL)
            IGNORE NULLS
            ORDER BY measurement_datetime DESC, measurement_id DESC
            LIMIT 1
        )[SAFE_OFFSET(0)] AS pupil_right_size_last,


        -- Linker pupil
        ARRAY_AGG(
            IF(pupil_side = 'left', pupil_size_mm, NULL)
            IGNORE NULLS
            ORDER BY measurement_datetime DESC, measurement_id DESC
            LIMIT 1
        )[SAFE_OFFSET(0)] AS pupil_left_size_last,


    FROM pupil_size

    GROUP BY
        visitId,
        attemptNumber
)

SELECT
    *,

    -- Gemiddelde pupilgrootte
    CASE
        WHEN pupil_right_size_last IS NOT NULL
         AND pupil_left_size_last IS NOT NULL
        THEN (pupil_right_size_last + pupil_left_size_last) / 2.0
    END AS pupil_size_mean_last,

    -- Verschil tussen linker- en rechterpupil
    CASE
        WHEN pupil_right_size_last IS NOT NULL
         AND pupil_left_size_last IS NOT NULL
        THEN ABS(pupil_right_size_last - pupil_left_size_last)
    END AS pupil_size_difference_last

FROM pupil_features
"""

pupil_size_features = client.query(sql).to_dataframe()

display(pupil_size_features)

epileptic seizure

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

seizures AS (
    SELECT
        c.visitId,
        c.attemptNumber,

        co.condition_start_datetime,
        co.condition_end_datetime,
        co.condition_occurrence_id,
        co.condition_concept_id,
        co.condition_source_value

    FROM cohort c

    JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` co
        ON c.visitId = co.visit_occurrence_id

    WHERE co.provider_id IS NOT NULL
        AND co.condition_concept_id IN (4197485)
        AND co.condition_start_datetime BETWEEN
            TIMESTAMP_SUB(c.extubationTime, INTERVAL 24 HOUR)
            AND c.extubationTime
)

SELECT
    c.visitId,
    c.attemptNumber,

    CASE
        WHEN COUNT(s.condition_occurrence_id) > 0 THEN 1
        ELSE 0
    END AS epileptic_seizure_24h,

    COUNT(s.condition_occurrence_id) AS epileptic_seizure_n_24h

FROM cohort c

LEFT JOIN seizures s
    ON c.visitId = s.visitId
    AND c.attemptNumber = s.attemptNumber

GROUP BY
    c.visitId,
    c.attemptNumber
"""

seizure_features = client.query(sql).to_dataframe()

display(seizure_features)

# feature engineering - Demographic

In [ ]:
demographic_variables = {
    "male": [8507],
    "female": [8532],
    "nationality": [4087925],
    "weight": [37206608],
    "kidney": [198124, 37018761],
    "cardiovascular": [134057, 4028244, 319835, 321588, 321319],
    "atrial_fibrillation": [313217, 4141360],
    "diabetes1": [201254],
    "hypertension": [316866],
}

Co morbidity

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
),

comorbidities AS (
    SELECT
        c.visitId,
        c.attemptNumber,
        co.condition_concept_id

    FROM cohort c
    JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.condition_occurrence` co
        ON c.visitId = co.visit_occurrence_id

    WHERE co.provider_id IS NOT NULL
      AND co.condition_concept_id IN (
          198124, 37018761,
          134057, 4028244, 319835, 321588, 321319,
          313217, 4141360,
          201254,
          316866
      )
      AND co.condition_start_datetime <= c.extubationTime
)

SELECT
    c.visitId,
    c.attemptNumber,

    MAX(CASE WHEN co.condition_concept_id IN (198124, 37018761)
        THEN 1 ELSE 0 END) AS comorbidity_kidney,

    MAX(CASE WHEN co.condition_concept_id IN (
        134057, 4028244, 319835, 321588, 321319
    ) THEN 1 ELSE 0 END) AS comorbidity_cardiovascular,

    MAX(CASE WHEN co.condition_concept_id IN (313217, 4141360)
        THEN 1 ELSE 0 END) AS comorbidity_atrial_fibrillation,

    MAX(CASE WHEN co.condition_concept_id = 201254
        THEN 1 ELSE 0 END) AS comorbidity_diabetes1,

    MAX(CASE WHEN co.condition_concept_id = 316866
        THEN 1 ELSE 0 END) AS comorbidity_hypertension

FROM cohort c
LEFT JOIN comorbidities co
    ON c.visitId = co.visitId
    AND c.attemptNumber = co.attemptNumber

GROUP BY
    c.visitId,
    c.attemptNumber
"""

df_comorbidity = client.query(sql).to_dataframe()

In [ ]:
comorbidity_columns = [
    "comorbidity_kidney",
    "comorbidity_cardiovascular",
    "comorbidity_atrial_fibrillation",
    "comorbidity_diabetes1",
    "comorbidity_hypertension"
]

df_comorbidity["comorbidity_count"] = (
    df_comorbidity[comorbidity_columns].sum(axis=1)
)

df_comorbidity["has_comorbidity"] = (
    df_comorbidity["comorbidity_count"] > 0
).astype(int)

display(df_comorbidity.head())

print(df_comorbidity[comorbidity_columns].sum())

print(
    df_comorbidity["comorbidity_count"].value_counts().sort_index()
)

gender

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT DISTINCT
    c.visitId,
    c.attemptNumber,
    p.gender_concept_id

FROM cohort c

LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
    ON c.visitId = vo.visit_occurrence_id

LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
    ON vo.person_id = p.person_id
"""

df_gender = client.query(sql).to_dataframe()

In [ ]:
df_gender["male"] = df_gender["gender_concept_id"].map({
    8507: True,
    8532: False
})

df_gender["female"] = df_gender["gender_concept_id"].map({
    8507: False,
    8532: True
})

In [ ]:
df = df.merge(
    df_gender[["visitId", "attemptNumber", "male", "female"]],
    on=["visitId", "attemptNumber"],
    how="left"
)

age

In [ ]:
sql = f"""
WITH cohort AS (
    SELECT *
    FROM UNNEST([{cohortRows}])
)

SELECT
    c.visitId,
    c.attemptNumber,
    DATE_DIFF(
        DATE(c.extubationTime),
        DATE(p.year_of_birth, 1, 1),
        YEAR
    ) AS age

FROM cohort c

LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.visit_occurrence` vo
    ON c.visitId = vo.visit_occurrence_id

LEFT JOIN `{DATASET_PROJECT_ID}.{DATASET_ID}.person` p
    ON vo.person_id = p.person_id
"""

df_age = client.query(sql).to_dataframe()

In [ ]:
display(df_age)
print(df_age.columns.tolist())